In [1]:
import yfinance as yf
import pandas as pd
import os
import openpyxl
from datetime import datetime as dt

In [18]:
# List of tickers ver celda mas abajo, con reservas de tickers
tickers = laggards # reemplazar el lado derecho por el nombre del array deseado
# Validate they all exist
for ticker in tickers:
    try:
        print("Ok\t",ticker,"\t", yf.Ticker(ticker).info["shortName"])
    except:
        print("Not ok\t", ticker)

Ok	 DLO 	 DLocal Limited
Ok	 NVO 	 Novo Nordisk A/S
Ok	 EVO.ST 	 Evolution AB
Ok	 GOOGL 	 Alphabet Inc.
Ok	 UNH 	 UnitedHealth Group Incorporated
Ok	 BABA 	 Alibaba Group Holding Limited
Ok	 PDD 	 PDD Holdings Inc.
Ok	 ASML 	 ASML Holding N.V. - New York Re


In [19]:
#### ESTE SCRIPT DEVUELVE UN DF CON TODOS LOS KPIS DE LOS TICKERS DE ARRIBA
import yfinance as yf
import pandas as pd

# Years to extract explicitly
YEARS = [2024, 2023, 2022, 2021, 2020]

# Define basic metrics
basic_metrics = {
    'Symbol':             lambda info: info.get('symbol'),
    'Currency':           lambda info: info.get('currency'),
    'Exchange':           lambda info: info.get('fullExchangeName'),
    'Latest Price':       lambda info: info.get('currentPrice'),
    'PE Ratio':           lambda info: info.get('trailingPE'),
    'EPS':                lambda info: info.get('trailingEps'),
    'TTM Dividend Yield (%)': lambda info: info.get('trailingAnnualDividendYield') or 0,
    'Dividend Rate':      lambda info: info.get('dividendRate', 0),
    'Debt to Equity': lambda info: info.get('debtToEquity') or 0,
    'Shares Outstanding': lambda info: info.get('sharesOutstanding', 0),
    'Market Cap (Billion $)': lambda info: info.get('marketCap', 0) / 1e9,
    'Volume': lambda info: info.get('volume'),
    '52-Week High':       lambda info: info.get('fiftyTwoWeekHigh'),
    '52-Week Low':        lambda info: info.get('fiftyTwoWeekLow'),
    'Dividend Yield L5Y (%)': lambda info: info.get('fiveYearAvgDividendYield') or 0,
    'Price/52-Week Low':  lambda info: (
        info.get('currentPrice') / info.get('fiftyTwoWeekLow')
        if info.get('currentPrice') and info.get('fiftyTwoWeekLow') else None
    ),
}

rows = []
for ticker in tickers:
    try:
        print(f"Getting data on {ticker}")
        stock = yf.Ticker(ticker)
        info = stock.info
    
        # Start each row with the company name
        row = {'Company': info.get('longName', ticker)}
    
        # Basic metrics
        for name, fn in basic_metrics.items():
            row[name] = fn(info)
    
        # Annual financials (Total Revenue & Net Income)
        fin = stock.financials
        rev_series = fin.loc['Total Revenue']
        net_series = fin.loc['Net Income']
    
        # Dividend history series
        div_series = stock.dividends
    
        # Explicit years for revenue, net income, dividends
        for year in YEARS:
            # Revenue in billions, or 0 if not found
            rev_vals = rev_series[rev_series.index.year == year]
            row[f'Revenue {year} (Billion $)'] = rev_vals.iloc[0] / 1e9 if not rev_vals.empty else 0
    
            # Net Income in billions, or 0 if not found
            net_vals = net_series[net_series.index.year == year]
            row[f'Net Income {year} (Billion $)'] = net_vals.iloc[0] / 1e9 if not net_vals.empty else 0
    
            # Dividends total for the year, zero if none
            year_divs = div_series[div_series.index.year == year].sum()
            row[f'Dividends {year}'] = year_divs
    
        rows.append(row)
    
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        continue

# Create DataFrame and sort by market cap
# Generate columns dynamically: Company, basics, then each year's metrics
columns = [
    'Company'
] + list(basic_metrics.keys()) + [
    f'Revenue {y} (Billion $)' for y in YEARS
] + [
    f'Net Income {y} (Billion $)' for y in YEARS
] + [
    f'Dividends {y}' for y in YEARS
]

df = pd.DataFrame(rows, columns=columns)
df["Payout ratio"] = df["Dividend Rate"]/df["EPS"]
df["Payout"] = df["Dividend Rate"]/df["EPS"]
if 'Market Cap (Billion $)' in df.columns:
    df = df.sort_values('Market Cap (Billion $)', ascending=False)

print(df)



Getting data on DLO
Getting data on NVO
Getting data on EVO.ST
Getting data on GOOGL
Getting data on UNH
Getting data on BABA
Getting data on PDD
Getting data on ASML
                           Company  Symbol Currency   Exchange  Latest Price  \
3                    Alphabet Inc.   GOOGL      USD   NasdaqGS        175.85   
1                 Novo Nordisk A/S     NVO      USD       NYSE         71.57   
7                ASML Holding N.V.    ASML      USD   NasdaqGS        804.54   
4  UnitedHealth Group Incorporated     UNH      USD       NYSE        304.69   
5    Alibaba Group Holding Limited    BABA      USD       NYSE        105.36   
2              Evolution AB (publ)  EVO.ST      SEK  Stockholm        772.20   
6                PDD Holdings Inc.     PDD      USD   NasdaqGS        104.84   
0                   DLocal Limited     DLO      USD   NasdaqGS         11.21   

    PE Ratio    EPS  TTM Dividend Yield (%)  Dividend Rate  Debt to Equity  \
3  19.648046   8.95               

In [21]:
# Save the DataFrame to an Excel file using openpyxl as the engine
today = dt.today().strftime('%Y%m%d')
df_sorted = df.sort_values(by=["Exchange", "PE Ratio"], ascending=[True, True])
file_path = fr'C:\Users\Julia\Downloads\stocks_data_{today}V2.xlsx'
df_sorted.to_excel(file_path, index=False)

In [17]:
# Reserva Tickers a Analizar
small_caps = ["ABCL","FUBO","HNST","HNST","SG","CDLR","KULR","ROOT","ONDS","LGCY","GAMB"] #https://x.com/Ashton_1nvests/status/1942958842242846974?t=-d0l27XlgJwduwtJvhP02Q&s=08
laggards = ["DLO","NVO","EVO.ST","GOOGL","UNH","BABA","PDD","ASML"] # https://x.com/TacticzH/status/1943044866935590989?t=2R3mHvU1x3GPLB3oIpv6QA&s=08

tickers_jp_brk = [
    # ITOCHU Corporation
    "8001.T",    # Tokyo
    "ITOCY",     # ADR (US)
    "IOC.F",    # Frankfurt — common local ticker

    # Marubeni Corporation
    "8002.T",    # Tokyo
    "MARUY",     # ADR (US)
    "MARA.F",     # Frankfurt — common for Mitsubishi Group

    # Mitsubishi Corporation
    "8058.T",    # Tokyo
    "MSBHF",     # ADR (US)
    "MBI.F",     # Frankfurt — typical
    "0Q0J.L",    # London (Mitsubishi Ord Shs) :contentReference[oaicite:1]{index=1}

    # Mitsui & Co., Ltd.
    "8031.T",    # Tokyo
    "MITSY",     # ADR (US)
    "MTS1.F",     # Frankfurt — typical

    # Sumitomo Corporation
    "8053.T",    # Tokyo
    "SSUMY",     # ADR (US)
    "SUMB.F",     # Frankfurt — typical

    # Sumitomo Mitsui Financial Group (added for LSE)
    "0LAF.L",    # London :contentReference[oaicite:2]{index=2}
]



# Tickers legacy, todo lo que veníamos monitoreando hasta el 2/6/24
tickers_legacy = [
    "2318.HK"
    , "AGN.AS"
    , "ASRNL.AS"
    , "ECMPA.AS"
    , "D05.SI"
    , "C6L.SI"
    , "C07.SI"   
    , "8316.T"    
    , "4503.T"   
    , "Z74.SI"
    , "LGEN.L"
    , "RIO"
    , "BN4.SI"
    , "BATS.L"
    , "PHNX.L"
    , "SDR.L"
    , "RKT"
    , "DGE.L"
    , "ULVR.L"
    , "UU.L"
    , "SVT"
    , "ASHM.L"
    , "AZN"
    , "RAT.L"
    , "ICG"
    , "ITRK.L"
    , "MONY.L"
    , "IGG.L"
    , "DCC.L"
    , "PETS"
    , "BNZL.L"
    , "POLR.L"
    , "CSN.L"
    , "BA.L"
    , "LIO.L"
    , "RWS.L"
    , "RS1.L" 
    , "SN.L"
    , "REL.L"
    , "IPX.L"
    , "TEP.L"
    , "AHT.L"
    , "OCN.L"
    , "CRDA.L"
    , "JHD.L"
    , "BOY.L"
    , "SGE.L"
    , "LSEG.L"
    , "MPE.L"
    , "SPX.L"
    , "SFR.L"
    , "U11.SI"
    , "F34.SI"
    , "S68.SI"
    , "BS6.SI"
    , "O39.SI"
    , "A6I.SG"
    , "C38U.SI"
    , "ADVANC.BK"
    , "CPALL.BK"
    , "TRUE.BK"
    , "DELTA.BK"
    , "CCET.BK"
    , "GULF.BK"
    , "AOT.BK"
    , "KBANK.BK"
#    , "SCB.BK"
    , "PTT.BK"
    , "BBL.BK" 
    , "BAY.BK"
    , "931.SG"
    , "AV.L"
    , "SBRE.L"
    , "HAL.AS"
    , "SHUR.BR"
    , "WDP.BR"
    , "WEHB.BR"
    , "XIOR.BR"
    , "MONT.BR"
    , "RET.BR"
    , "CPINV.BR"
    , "MGCI.L"
    , "RECI.L"
    , "DUKE.L"
    , "SEQI.L"
    , "NESF.L"
    , "IPX.L"
    , "PMI.L"
    , "TRIG.L"
    , "SUPR.L"
    , "SMIF.L"
    , "GCP.L"
    , "SEIT.L"
    , "GSF.L"
    , "FSFL.L"
    , "UKW.L"
    , "ENRG.L"
    , "EXO.AS"
    , "ITM.MI"
    , "FFP.F"
    , "WIS.MU"
]



# Tuit https://x.com/whu32/status/1929072703790530595?t=o0Zd2s_EHJpEMm-HwYr3nw&s=08
tickers_tuitwhu32 = [
    "PHNX.L",   # Phoenix Group Holdings
    "SUPR.L",   # Atrato Partners Supermarket Income REIT
    "SREI.L",   # Schroder Real Estate Investment Trust
    "LGEN.L",   # Legal & General Group
    "AEWU.L",   # AEW UK Investment Management LLP
    "CREI.L",   # Custodian Property Income REIT
    "RECI.L",   # Cheyne Real Estate Credit Investments Ltd
    "CSN.L",   # Chesnara
    "TFIF.L",   # TwentyFour Select Monthly Income Fund
    "LMP.L",    # LondonMetric Property PLC
    "MNG.L",    # M&G
    "NBS.L",    # Newcastle Building Society
    "ASLI.L",   # abrdn European Logistics Income
    "TFIF.L",   # TwentyFour Income Fund
    "BLND.L",   # British Land Company
    "PCTN.L",   # Picton Property Income
    "LAND.L",   # Land Securities Group
    "SGRO.L",   # Segro
]

# Singapore tickers
tickers_sgx = [
  "1A0.SI",
  "1A1.SI",
  "1AZ.SI",
  "1B0.SI",
  "1B1.SI",
  "1B6.SI",
  "1C0.SI",
  "1D0.SI",
  "1D1.SI",
  "1D3.SI",
  "1D4.SI",
  "1D5.SI",
  "1E3.SI",
  "1F0.SI",
  "1F1.SI",
  "1F2.SI",
  "1F3.SI",
  "1H2.SI",
  "1H3.SI",
  "1H8.SI",
  "1J0.SI",
  "1J4.SI",
  "1J5.SI",
  "1J7.SI",
  "1L2.SI",
  "1MZ.SI",
  "1R6.SI",
  "1V3.SI",
  "1Y1.SI",
  "40B.SI",
  "40E.SI",
  "40F.SI",
  "40N.SI",
  "40T.SI",
  "40V.SI",
  "40W.SI",
  "41B.SI",
  "41F.SI",
  "41H.SI",
  "41O.SI",
  "41T.SI",
  "42C.SI",
  "42E.SI",
  "42F.SI",
  "42L.SI",
  "42N.SI",
  "42R.SI",
  "42S.SI",
  "42T.SI",
  "42W.SI",
  "42Z.SI",
  "43A.SI",
  "43B.SI",
  "43E.SI",
  "43F.SI",
  "43Q.SI",
  "49B.SI",
  "500.SI",
  "504.SI",
  "505.SI",
  "508.SI",
  "532.SI",
  "533.SI",
  "53W.SI",
  "540.SI",
  "541.SI",
  "543.SI",
  "544.SI",
  "546.SI",
  "554.SI",
  "558.SI",
  "564.SI",
  "566.SI",
  "569.SI",
  "570.SI",
  "575.SI",
  "579.SI",
  "580.SI",
  "581.SI",
  "583.SI",
  "584.SI",
  "585.SI",
  "594.SI",
  "595.SI",
  "596.SI",
  "5AB.SI",
  "5AE.SI",
  "5AI.SI",
  "5AL.SI",
  "5AU.SI",
  "5BI.SI",
  "5BS.SI",
  "5CF.SI",
  "5CR.SI",
  "5CT.SI",
  "5DD.SI",
  "5DM.SI",
  "5DN.SI",
  "5DO.SI",
  "5DP.SI",
  "5DS.SI",
  "5DX.SI",
  "5E2.SI",
  "5EB.SI",
  "5EF.SI",
  "5EG.SI",
  "5EN.SI",
  "5EV.SI",
  "5EW.SI",
  "5F4.SI",
  "5F7.SI",
  "5FW.SI",
  "5FX.SI",
  "5G1.SI",
  "5G2.SI",
  "5G3.SI",
  "5G4.SI",
  "5G9.SI",
  "5GD.SI",
  "5GI.SI",
  "5GJ.SI",
  "5GZ.SI",
  "5HG.SI",
  "5HH.SI",
  "5HT.SI",
  "5HV.SI",
  "5I1.SI",
  "5I4.SI",
  "5IC.SI",
  "5IE.SI",
  "5IF.SI",
  "5IG.SI",
  "5JK.SI",
  "5JS.SI",
  "5KI.SI",
  "5LE.SI",
  "5LY.SI",
  "5MD.SI",
  "5ML.SI",
  "5MZ.SI",
  "5NF.SI",
  "5NV.SI",
  "5OC.SI",
  "5OI.SI",
  "5OQ.SI",
  "5OR.SI",
  "5OX.SI",
  "5PC.SI",
  "5PD.SI",
  "5PF.SI",
  "5PO.SI",
  "5QR.SI",
  "5QT.SI",
  "5QY.SI",
  "5RA.SI",
  "5RC.SI",
  "5RE.SI",
  "5RF.SI",
  "5SO.SI",
  "5SR.SI",
  "5SY.SI",
  "5TI.SI",
  "5TJ.SI",
  "5TP.SI",
  "5TT.SI",
  "5UA.SI",
  "5UF.SI",
  "5UL.SI",
  "5UX.SI",
  "5VC.SI",
  "5VI.SI",
  "5VJ.SI",
  "5VP.SI",
  "5VS.SI",
  "5WA.SI",
  "5WF.SI",
  "5WG.SI",
  "5WH.SI",
  "5WJ.SI",
  "5WV.SI",
  "600.SI",
  "7QQS.SI",
  "8A1.SI",
  "8AZ.SI",
  "8K7.SI",
  "8U7U.SI",
  "8YY.SI",
  "9A4U.SI",
  "9CI.SI",
  "9G2.SI",
  "9I7.SI",
  "9QX.SI",
  "A04.SI",
  "A05.SI",
  "A17U.SI",
  "A26.SI",
  "A30.SI",
  "A31.SI",
  "A33.SI",
  "A34.SI",
  "A35.SI",
  "A50.SI",
  "A52.SI",
  "A55.SI",
  "A78.SI",
  "A7RU.SI",
  "A93.SI",
  "A94.SI",
  "AAJ.SI",
  "ACV.SI",
  "ADN.SI",
  "AFUS.SI",
  "AGS.SI",
  "AIY.SI",
  "AJ2.SI",
  "AJBU.SI",
  "AOF.SI",
  "AP4.SI",
  "AU8U.SI",
  "AVX.SI",
  "AW9U.SI",
  "AWC.SI",
  "AWG.SI",
  "AWI.SI",
  "AWK.SI",
  "AWM.SI",
  "AWV.SI",
  "AWX.SI",
  "AWZ.SI",
  "AYN.SI",
  "AYV.SI",
  "AZA.SI",
  "AZG.SI",
  "AZR.SI",
  "AZT.SI",
  "B0Z.SI",
  "B26.SI",
  "B28.SI",
  "B49.SI",
  "B58.SI",
  "B61.SI",
  "B69.SI",
  "B73.SI",
  "B81R.SI",
  "B9S.SI",
  "BAC.SI",
  "BAI.SI",
  "BAZ.SI",
  "BBP.SI",
  "BBW.SI",
  "BCD.SI",
  "BCV.SI",
  "BCY.SI",
  "BCZ.SI",
  "BDA.SI",
  "BDR.SI",
  "BDU.SI",
  "BDX.SI",
  "BEC.SI",
  "BEH.SI",
  "BEI.SI",
  "BEW.SI",
  "BEZ.SI",
  "BFI.SI",
  "BFK.SI",
  "BFT.SI",
  "BFU.SI",
  "BGO.SI",
  "BHD.SI",
  "BHK.SI",
  "BHU.SI",
  "BIP.SI",
  "BIX.SI",
  "BJD.SI",
  "BJGS.SI",
  "BJHS.SI",
  "BJIS.SI",
  "BJV.SI",
  "BJZ.SI",
  "BKA.SI",
  "BKK.SI",
  "BKV.SI",
  "BKW.SI",
  "BKX.SI",
  "BKZ.SI",
  "BLH.SI",
  "BLR.SI",
  "BLS.SI",
  "BLU.SI",
  "BLZ.SI",
  "BMGU.SI",
  "BMT.SI",
  "BN2.SI",
  "BN4.SI",
  "BNE.SI",
  "BPF.SI",
  "BQC.SI",
  "BQD.SI",
  "BQF.SI",
  "BQM.SI",
  "BQN.SI",
  "BQP.SI",
  "BRD.SI",
  "BRS.SI",
  "BS6.SI",
  "BSL.SI",
  "BTE.SI",
  "BTF.SI",
  "BTG.SI",
  "BTJ.SI",
  "BTM.SI",
  "BTOU.SI",
  "BTP.SI",
  "BTX.SI",
  "BTY.SI",
  "BUOU.SI",
  "BVA.SI",
  "BVQ.SI",
  "BWCU.SI",
  "BWM.SI",
  "BXE.SI",
  "BYI.SI",
  "BYJ.SI",
  "C04.SI",
  "C05.SI",
  "C06.SI",
  "C07.SI",
  "C09.SI",
  "C13.SI",
  "C2PU.SI",
  "C33.SI",
  "C38U.SI",
  "C41.SI",
  "C52.SI",
  "C6L.SI",
  "C70.SI",
  "C76.SI",
  "C8R.SI",
  "C9Q.SI",
  "CC3.SI",
  "CDVZ.SI",
  "CEDU.SI",
  "CFA.SI",
  "CHJ.SI",
  "CHZ.SI",
  "CIN.SI",
  "CJLU.SI",
  "CJN.SI",
  "CLN.SI",
  "CLR.SI",
  "CMGS.SI",
  "CMOU.SI",
  "CNE.SI",
  "COI.SI",
  "CRPU.SI",
  "CTO.SI",
  "CWBU.SI",
  "CWCU.SI",
  "CXS.SI",
  "CXU.SI",
  "CY6U.SI",
  "CYB.SI",
  "CYC.SI",
  "CYW.SI",
  "CYX.SI",
  "D01.SI",
  "D03.SI",
  "D05.SI",
  "D07.SI",
  "D5IU.SI",
  "D8DU.SI",
  "DCRU.SI",
  "DHLU.SI",
  "DM0.SI",
  "DRX.SI",
  "DU4.SI",
  "E27.SI",
  "E28.SI",
  "E3B.SI",
  "E5H.SI",
  "E6R.SI",
  "E9L.SI",
  "EAA.SI",
  "EAU.SI",
  "EB5.SI",
  "EH5.SI",
  "EHG.SI",
  "EMI.SI",
  "ENV.SI",
  "ER0.SI",
  "ES3.SI",
  "ESG.SI",
  "ESU.SI",
  "EVD.SI",
  "EVS.SI",
  "F03.SI",
  "F10.SI",
  "F13.SI",
  "F17.SI",
  "F1E.SI",
  "F34.SI",
  "F83.SI",
  "F86.SI",
  "F99.SI",
  "F9D.SI",
  "FQ7.SI",
  "FRQ.SI",
  "G07.SI",
  "G0I.SI",
  "G13.SI",
  "G1N.SI",
  "G20.SI",
  "G3B.SI",
  "G50.SI",
  "G92.SI",
  "GEH.SI",
  "GRE.SI",
  "GRN.SI",
  "GRO.SI",
  "GRQ.SI",
  "GRU.SI",
  "GSD.SI",
  "GU5.SI",
  "H02.SI",
  "H07.SI",
  "H12.SI",
  "H13.SI",
  "H15.SI",
  "H18.SI",
  "H1N.SI",
  "H20.SI",
  "H22.SI",
  "H30.SI",
  "H78.SI",
  "HBBD.SI",
  "HBND.SI",
  "HD9.SI",
  "HKB.SI",
  "HLS.SI",
  "HMN.SI",
  "HMTD.SI",
  "HPAD.SI",
  "HQU.SI",
  "HSHD.SI",
  "HSS.SI",
  "HST.SI",
  "HTCD.SI",
  "HXXD.SI",
  "HYDD.SI",
  "i06.SI",
  "i07.SI",
  "i11.SI",
  "i49.SI",
  "I98.SI",
  "ICH.SI",
  "ICM.SI",
  "ICU.SI",
  "INC.SI",
  "IX2.SI",
  "J03.SI",
  "J2T.SI",
  "J36.SI",
  "J69U.SI",
  "J85.SI",
  "JJJ.SI",
  "JK8.SI",
  "JLB.SI",
  "JUS.SI",
  "JYEU.SI",
  "K03.SI",
  "K29.SI",
  "K3MD.SI",
  "K3RD.SI",
  "K3SD.SI",
  "K6S.SI",
  "K71U.SI",
  "K75.SI",
  "KJ5.SI",
  "KJ7.SI",
  "KUH.SI",
  "KUO.SI",
  "KUX.SI",
  "KV4.SI",
  "KYB.SI",
  "L02.SI",
  "L19.SI",
  "L23.SI",
  "L38.SI",
  "LCS.SI",
  "LCU.SI",
  "LG9.SI",
  "LIW.SI",
  "LJ3.SI",
  "LMS.SI",
  "LS9.SI",
  "LSS.SI",
  "LSU.SI",
  "LUY.SI",
  "LVR.SI",
  "LYY.SI",
  "M01.SI",
  "M03.SI",
  "M04.SI",
  "M05.SI",
  "M11.SI",
  "M14.SI",
  "M15.SI",
  "M1GU.SI",
  "M44U.SI",
  "M62.SI",
  "MBH.SI",
  "MCN.SI",
  "MCS.SI",
  "ME8U.SI",
  "MF6.SI",
  "MIJ.SI",
  "MMS.SI",
  "MMT.SI",
  "MR7.SI",
  "MV4.SI",
  "MXNU.SI",
  "MZH.SI",
  "N01.SI",
  "N02.SI",
  "N08.SI",
  "N0Z.SI",
  "N2H.SI",
  "N2IU.SI",
  "N32.SI",
  "N5YD.SI",
  "N6DD.SI",
  "N6FD.SI",
  "N6M.SI",
  "NC2.SI",
  "NEX.SI",
  "NHD.SI",
  "NIO.SI",
  "NPL.SI",
  "NPW.SI",
  "NR7.SI",
  "NS8U.SI",
  "NXR.SI",
  "O08.SI",
  "O10.SI",
  "O39.SI",
  "O5RU.SI",
  "O6Z.SI",
  "O87.SI",
  "O9A.SI",
  "O9E.SI",
  "O9P.SI",
  "OAJ.SI",
  "ODBU.SI",
  "OL9S.SI",
  "OMK.SI",
  "OTS.SI",
  "OTX.SI",
  "OU8.SI",
  "OV8.SI",
  "OVQ.SI",
  "OVS",
  "OXMU.SI",
  "OYY.SI",
  "P11.SI",
  "P15.SI",
  "P34.SI",
  "P36.SI",
  "P40U.SI",
  "P52.SI",
  "P5P.SI",
  "P74.SI",
  "P7VU.SI",
  "P8A.SI",
  "P8Z.SI",
  "P9D.SI",
  "PA3.SI",
  "PCT.SI",
  "PH0.SI",
  "PH1S.SI",
  "PPC.SI",
  "PRH.SI",
  "PU6D.SI",
  "Q01.SI",
  "Q0F.SI",
  "Q0X.SI",
  "Q5T.SI",
  "QC7.SI",
  "QES.SI",
  "QK9.SI",
  "QL2.SI",
  "QL3.SI",
  "QNS.SI",
  "QR9.SI",
  "QS0.SI",
  "QS9.SI",
  "QZG.SI",
  "R14.SI",
  "R1NS.SI",
  "RC5.SI",
  "RCU.SI",
  "RDR.SI",
  "RE4.SI",
  "RQ1.SI",
  "RXS.SI",
  "S07.SI",
  "S08.SI",
  "S19.SI",
  "S20.SI",
  "S23.SI",
  "S27.SI",
  "S29.SI",
  "S2D.SI",
  "S35.SI",
  "S3N.SI",
  "S41.SI",
  "S44.SI",
  "S45U.SI",
  "S56.SI",
  "S58.SI",
  "S59.SI",
  "S61.SI",
  "S63.SI",
  "S68.SI",
  "S69.SI",
  "S71.SI",
  "S7OU.SI",
  "S85.SI",
  "S9B.SI",
  "SCY.SI",
  "SEJ.SI",
  "SES.SI",
  "SGR.SI",
  "SHD.SI",
  "SJY.SI",
  "SK3.SI",
  "SK6U.SI",
  "SO7.SI",
  "SQQ.SI",
  "SQU.SI",
  "SRT.SI",
  "SRU.SI",
  "SSS.SI",
  "SSU.SI",
  "STC.SI",
  "STG.SI",
  "T09.SI",
  "T12.SI",
  "T13.SI",
  "T14.SI",
  "T15.SI",
  "T24.SI",
  "T41.SI",
  "T43.SI",
  "T4B.SI",
  "T55.SI",
  "T6I.SI",
  "T82U.SI",
  "T8FS.SI",
  "TADD.SI",
  "TATD.SI",
  "TCPD.SI",
  "TCU.SI",
  "TDED.SI",
  "TID.SI",
  "TKKD.SI",
  "TPED.SI",
  "TQ5.SI",
  "TS0U.SI",
  "TSCD.SI",
  "TSH.SI",
  "TVV.SI",
  "TWL.SI",
  "U06.SI",
  "U09.SI",
  "U10.SI",
  "U11.SI",
  "U13.SI",
  "U14.SI",
  "U77.SI",
  "U96.SI",
  "U9E.SI",
  "UD1U.SI",
  "UD2.SI",
  "UIX.SI",
  "URR.SI",
  "UUK.SI",
  "UV1.SI",
  "V03.SI",
  "V2Y.SI",
  "V3M.SI",
  "V5Q.SI",
  "V7R.SI",
  "V8Y.SI",
  "VC2.SI",
  "VI2.SI",
  "VIN.SI",
  "VND.SI",
  "VNM.SI",
  "VVL.SI",
  "W05.SI",
  "WJ9.SI",
  "WJP.SI",
  "WKS.SI",
  "WPC.SI",
  "WVJ.SI",
  "X5N.SI",
  "XCF.SI",
  "XHV.SI",
  "XJB.SI",
  "XVG.SI",
  "XWA.SI",
  "XZL.SI",
  "Y03.SI",
  "Y06.SI",
  "Y35.SI",
  "Y3D.SI",
  "Y8E.SI",
  "Y92.SI",
  "YF8.SI",
  "YK9.SI",
  "YLD.SI",
  "YLU.SI",
  "YYB.SI",
  "YYN.SI",
  "YYR.SI",
  "YYY.SI",
  "Z25.SI",
  "Z4D.SI",
  "Z59.SI",
  "Z74.SI",
  "Z77.SI",
  "ZB9.SI",
  "ZBVR.SI",
  "ZHD.SI",
  "ZHS.SI",
  "ZHY.SI",
  "ZKX.SI",
  "ZXY.SI"
]

tickers_uk_divs = [
    "MNG.L"    # M&G,
    ,"LGEN.L"   # Legal & General,
    ,"PHNX.L"   # Phoenix Group,
    ,"TW.L"     # Taylor Wimpey,
    ,"BATS.L"   # British American Tobacco,
    ,"RIO.L"    # Rio Tinto,
    ,"BP.L"     # BP,
    ,"WPP.L"    # WPP Group,
    ,"LAND.L"   # Land Securities,
    ,"SDR.L"    # Schroders
]

tickers_tuit_emf_junio25 =  [
    "ULVR.L",     # Unilever (London)
    "GILD",       # Gilead Sciences
    "NOVN.SW",    # Novartis (Switzerland)
    "BLX",        # Bladex
    "AMGN",       # Amgen
    "021240.KQ",  # Coway (KOSDAQ)
    "HCLTECH.NS", # HCL Technologies (NSE India)
    "TE.PA",      # Technip Energies (Paris)
    "ENEL.MI",    # Enel (Milan)
    "IMB.L",      # Imperial Brands (London)
    "ZURN.SW",    # Zurich Insurance (Switzerland)
    "UNM",        # Unum
    "MUV2.DE",    # Munich Re (Germany)
    "KO",         # Coca-Cola
    "MDT",        # Medtronic
    "RDN",        # Radian
    "BN.PA",      # Danone (Paris)
    "TSCO.L",     # Tesco (London)
    "OFG"         # OFG Bancorp
]



In [ ]:
valid_tickers = []

for ticker in tickers:
    try:
        name = yf.Ticker(ticker).info["shortName"]
        print("Ok\t", ticker, "\t", name)
        valid_tickers.append(f'"{ticker}"')  # formato para array python
    except:
        print("Not ok\t", ticker)

# Imprimí el array final ya listo para copiar/pegar
print("\nvalid_tickers = [\n  " + ",\n  ".join(valid_tickers) + "\n]")